In [18]:
data = """1.0240 1.5360 2.0480 3.0720 4.0960 6.1440 8.1920 9.2160 12.2880 16.3840 24.5760
2.0480 3.0720 4.0960 6.1440 8.1920 12.2880 16.3840 18.4320 24.5760 36.8640 49.1520
4.0960 6.1440 8.1920 12.2880 16.3840 24.5760 32.7680 36.8640 49.1520
5.6488 8.4672 11.2896 16.9344 22.5792 33.8688 45.1584
6.1440 9.2160 12.2880 18.4320 24.5760 36.8640 49.1520
11.2896 16.9344 22.5792 33.8688 45.1584
12.2880 18.4320 24.5760 36.8640 49.1520
22.5792 33.8688 45.1584
24.5760 36.8640 49.1520
24.5760 49.1520"""

tokens = data.split()

for i in range(len(tokens)):
    tokens[i] = float(tokens[i])

# print(tokens)

In [19]:
def calculate_pll_setup(output_freq_mhz):

    device_limits = {
        "GW2AR-18 C8/I7": {
            "comment": "untested",
            "pll_name": "rPLL",
            "pfd_min": 3,
            "pfd_max": 500,
            "vco_min": 500,
            "vco_max": 1250,
            "clkout_min": 3.90625,
            "clkout_max": 625,
        },
    }

    limits = device_limits["GW2AR-18 C8/I7"]
    setup = {}

    FCLKIN = 27
    min_diff = FCLKIN

    for IDIV_SEL in range(64):
        for FBDIV_SEL in range(64):
            for ODIV_SEL in [2, 4, 8, 16, 32, 48, 64, 80, 96, 112, 128]:
                PFD = FCLKIN / (IDIV_SEL + 1)
                if not (limits["pfd_min"] <= PFD <= limits["pfd_max"]):
                    continue
                CLKOUT = FCLKIN * (FBDIV_SEL + 1) / (IDIV_SEL + 1)
                if not (limits["clkout_min"] < CLKOUT < limits["clkout_max"]):
                    continue
                VCO = (FCLKIN * (FBDIV_SEL + 1) * ODIV_SEL) / (IDIV_SEL + 1)
                if not (limits["vco_min"] < VCO < limits["vco_max"]):
                    continue
                diff = abs(output_freq_mhz - CLKOUT)
                rel_error = diff/output_freq_mhz
                if diff < min_diff:
                    min_diff = diff
                    setup = {
                        "DESIRED_CLKOUT": output_freq_mhz,
                        "CLKOUT": CLKOUT,
                        "IDIV_SEL": IDIV_SEL,
                        "FBDIV_SEL": FBDIV_SEL,
                        "ODIV_SEL": ODIV_SEL,
                        "PFD": PFD,
                        "VCO": VCO,
                        "ERROR": diff,
                        "REL_ERROR": rel_error
                    }
    return setup

In [21]:
results = []
for token in tokens:
    results.append(calculate_pll_setup(token))

sorted_results = sorted(results, key=lambda x: x['REL_ERROR'])

for sorted_result in sorted_results:
    print(sorted_result)

{'DESIRED_CLKOUT': 16.9344, 'CLKOUT': 16.875, 'IDIV_SEL': 7, 'FBDIV_SEL': 4, 'ODIV_SEL': 32, 'PFD': 3.375, 'VCO': 540.0, 'ERROR': 0.05940000000000012, 'REL_ERROR': 0.0035076530612244967}
{'DESIRED_CLKOUT': 22.5792, 'CLKOUT': 22.5, 'IDIV_SEL': 5, 'FBDIV_SEL': 4, 'ODIV_SEL': 32, 'PFD': 4.5, 'VCO': 720.0, 'ERROR': 0.07920000000000016, 'REL_ERROR': 0.0035076530612244967}
{'DESIRED_CLKOUT': 33.8688, 'CLKOUT': 33.75, 'IDIV_SEL': 3, 'FBDIV_SEL': 4, 'ODIV_SEL': 16, 'PFD': 6.75, 'VCO': 540.0, 'ERROR': 0.11880000000000024, 'REL_ERROR': 0.0035076530612244967}
{'DESIRED_CLKOUT': 45.1584, 'CLKOUT': 45.0, 'IDIV_SEL': 2, 'FBDIV_SEL': 4, 'ODIV_SEL': 16, 'PFD': 9.0, 'VCO': 720.0, 'ERROR': 0.15840000000000032, 'REL_ERROR': 0.0035076530612244967}
{'DESIRED_CLKOUT': 16.9344, 'CLKOUT': 16.875, 'IDIV_SEL': 7, 'FBDIV_SEL': 4, 'ODIV_SEL': 32, 'PFD': 3.375, 'VCO': 540.0, 'ERROR': 0.05940000000000012, 'REL_ERROR': 0.0035076530612244967}
{'DESIRED_CLKOUT': 22.5792, 'CLKOUT': 22.5, 'IDIV_SEL': 5, 'FBDIV_SEL': 4, 